# Phase 6: Advanced Modeling — Improving Your Kaggle Score

## Objectives

In this phase, you will:

1. Understand why Linear Regression scored only 0.19 on Kaggle and what went wrong
2. Learn what RMSLE is and why log-transforming the target is critical
3. Apply regularized models (Ridge, Lasso, ElasticNet) to fix overfitting
4. Train tree-based models (Random Forest, XGBoost, LightGBM) that capture non-linear patterns
5. Compare all models with proper cross-validation
6. Blend the best models into a final ensemble for the highest score
7. Generate a Kaggle submission and measure improvement

## What We Missed in Phase 5 and Why It Matters

Our Phase 5 Linear Regression homework had several critical limitations:

- **No log-transform on SalePrice**: Kaggle scores this competition using RMSLE (Root Mean Squared Log Error), not RMSE. By predicting raw prices, large houses dominated the error. Working in log-space converts RMSLE into plain RMSE, which our models optimize directly.
- **No regularization**: After one-hot encoding and feature engineering, we had 200+ features. Plain Linear Regression fits ALL of them with no penalty, causing coefficients to explode — classic overfitting.
- **Only one model**: Linear Regression assumes a straight-line relationship between features and price. In reality, house prices have non-linear patterns (e.g., OverallQual has an exponential effect on price, not linear).
- **No cross-validation**: A single 80/20 split is unreliable. One unlucky split can give a misleadingly good or bad score.
- **No model blending**: Different models make different mistakes. Averaging predictions from multiple models cancels out individual errors.

## TODO Checklist

- [ ] Task 1: Import libraries
- [ ] Task 2: Load data and log-transform the target
- [ ] Task 3: Preprocess features for modeling
- [ ] Task 4: Set up cross-validation framework
- [ ] Task 5: Train Ridge, Lasso, and ElasticNet
- [ ] Task 6: Understand and train Random Forest
- [ ] Task 7: Understand and train XGBoost
- [ ] Task 8: Understand and train LightGBM
- [ ] Task 9: Compare all models
- [ ] Task 10: Blend the best models
- [ ] Task 11: Generate Kaggle submission
- [ ] Task 12: Summary and reflection

## Instructions

Apply the advanced modeling techniques below to beat your Phase 5 score. Follow each task step-by-step and fill in the TODO sections with your code.

---
## Why Linear Regression Is Limited for This Problem

Linear Regression makes these assumptions:

1. **Linearity**: The relationship between every feature and price is a straight line. But consider OverallQual: a jump from quality 8→10 increases price far more than 2→4. This is a non-linear (exponential) effect that Linear Regression cannot capture.

2. **No multicollinearity**: Features should be independent. But `TotalSF` and `GrLivArea` and `1stFlrSF` are all highly correlated. With 200+ features after encoding, many are redundant, causing coefficients to become unstable and huge.

3. **Equal feature treatment**: Linear Regression gives every feature a coefficient with no penalty. With hundreds of features, it overfits by memorizing noise in the training data.

### What we need instead:

| Problem | Solution | Model |
|---|---|---|
| Coefficients too large / overfitting | Add penalty to shrink coefficients | **Ridge, Lasso, ElasticNet** |
| Too many useless features | Automatically zero out weak features | **Lasso** |
| Non-linear relationships | Use trees that split data into regions | **Random Forest, XGBoost, LightGBM** |
| Single model has blind spots | Combine multiple models | **Ensemble / Blending** |

---
## Task 1: Import Required Libraries

In [ ]:
# TODO: Import core libraries
# - import pandas as pd
# - import numpy as np
# - import matplotlib.pyplot as plt
# - import seaborn as sns
# - import warnings
# - warnings.filterwarnings('ignore')

# TODO: Import sklearn tools
# - from sklearn.model_selection import train_test_split, cross_val_score, KFold
# - from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
# - from sklearn.preprocessing import StandardScaler

# TODO: Import regularized linear models
# - from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression

# TODO: Import tree-based models
# - from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
# - from xgboost import XGBRegressor
# - from lightgbm import LGBMRegressor

# TODO: Set visualization style
# - sns.set_style('whitegrid')
# - plt.rcParams['figure.figsize'] = (12, 6)

# TODO: Print success message

---
## Task 2: Load Data and Log-Transform the Target

### Why log-transform SalePrice?

Kaggle evaluates this competition using **RMSLE** (Root Mean Squared **Log** Error):

$$\text{RMSLE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} \left( \log(\hat{y}_i + 1) - \log(y_i + 1) \right)^2}$$

If we apply `log(1 + y)` to our target **before** training, then RMSLE becomes plain RMSE — which is exactly what our models minimize. This is a free improvement.

Additionally, SalePrice is right-skewed (most houses are cheap, a few are very expensive). Log-transform makes the distribution more normal, which helps linear models.

**Important:** After predicting, we must reverse this with `np.expm1()` (which computes `exp(x) - 1`) to get back to actual dollar prices.

In [ ]:
# TODO: Load feature-engineered training data
# - Use pd.read_csv() to load 'Solutions/train_feature_engineered.csv'
# - Store in variable 'df'
# - Print shape

# TODO: Separate features and target
# - X = df.drop('SalePrice', axis=1)
# - y_raw = df['SalePrice']  (keep raw prices for later comparison)

# TODO: Log-transform the target
# - y = np.log1p(y_raw)  (this computes log(1 + price))
# - Print y_raw.describe() vs y.describe() to see the difference

# TODO: Visualize the transformation
# - Create 1x2 subplots
# - Left: histogram of y_raw with title 'Original SalePrice'
# - Right: histogram of y with title 'Log-Transformed SalePrice'
# - Notice how the right plot is more symmetric (normal-shaped)

---
## Task 3: Preprocess Features for Modeling

Before training models, we need to:
1. Handle any remaining categorical columns (one-hot encode them)
2. Drop the `Id` column (it's not a feature)
3. Fill any missing values
4. Scale features for regularized linear models

In [ ]:
# TODO: Drop the Id column if present
# - if 'Id' in X.columns: X = X.drop('Id', axis=1)

# TODO: Identify categorical columns
# - cat_cols = X.select_dtypes(include=['object']).columns.tolist()
# - Print how many categorical columns remain
# - Print their names

# TODO: One-hot encode categorical columns
# - X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
# - Print new shape after encoding

# TODO: Fill any remaining missing values with 0
# - X = X.fillna(0)
# - Verify: print X.isnull().sum().sum() (should be 0)

# TODO: Print final feature count

In [ ]:
# TODO: Split data into training and testing sets
# - Use: X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# - Print shapes of all four sets

# TODO: Scale features (needed for Ridge, Lasso, ElasticNet)
# - scaler = StandardScaler()
# - X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
# - X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)
# - Note: fit on training data, transform on test data (never fit on test!)
# - Note: tree-based models (RF, XGBoost, LightGBM) do NOT need scaling

---
## Task 4: Set Up Cross-Validation Framework

### Why cross-validation instead of a single train/test split?

A single 80/20 split is like grading a student on one test — they might get lucky or unlucky. **5-Fold Cross-Validation** splits data into 5 parts, trains 5 times (each time holding out a different part), and averages the scores. This gives a much more reliable estimate of how the model will perform on Kaggle.

```
Fold 1: [TEST] [Train] [Train] [Train] [Train]  → Score 1
Fold 2: [Train] [TEST] [Train] [Train] [Train]  → Score 2
Fold 3: [Train] [Train] [TEST] [Train] [Train]  → Score 3
Fold 4: [Train] [Train] [Train] [TEST] [Train]  → Score 4
Fold 5: [Train] [Train] [Train] [Train] [TEST]  → Score 5

Final Score = Average(Score 1..5)  ±  Std(Score 1..5)
```

In [ ]:
# TODO: Set up K-Fold cross-validation
# - kf = KFold(n_splits=5, shuffle=True, random_state=42)

# TODO: Create a helper function to evaluate models using cross-validation
# def rmse_cv(model, X, y):
#     scores = cross_val_score(model, X, y, scoring='neg_root_mean_squared_error', cv=kf)
#     return -scores  (negate because sklearn returns negative RMSE)
#
# Usage: scores = rmse_cv(model, X_train_scaled, y_train)
#        print(f"RMSE: {scores.mean():.4f} (+/- {scores.std():.4f})")

# TODO: Create a dictionary to store results from all models
# - results = {}  (we'll fill this as we train each model)

# TODO: Print confirmation

---
## Task 5: Regularized Linear Models — Ridge, Lasso, ElasticNet

### What went wrong with plain Linear Regression?

Linear Regression minimizes only the prediction error (MSE). With 200+ features, it has too much freedom — it assigns large coefficients to noisy features and overfits.

**Regularization** adds a penalty term that forces the model to keep coefficients small:

### Ridge (L2 Regularization)

$$\text{Loss} = \text{MSE} + \alpha \sum_{j=1}^{p} \beta_j^2$$

- Adds the **sum of squared coefficients** as a penalty
- Shrinks all coefficients toward zero but never exactly to zero
- Good when many features contribute a little bit each
- `alpha` controls penalty strength: higher alpha = smaller coefficients

### Lasso (L1 Regularization)

$$\text{Loss} = \text{MSE} + \alpha \sum_{j=1}^{p} |\beta_j|$$

- Adds the **sum of absolute coefficients** as a penalty
- Can shrink coefficients exactly to zero — automatic feature selection
- Good when only a few features are truly important
- With 200+ features, Lasso will likely zero out 100+ of them

### ElasticNet (L1 + L2 Combined)

$$\text{Loss} = \text{MSE} + \alpha \left( r \sum |\beta_j| + \frac{1-r}{2} \sum \beta_j^2 \right)$$

- Combines Ridge and Lasso: `l1_ratio` (r) controls the mix
- `l1_ratio=1.0` → pure Lasso, `l1_ratio=0.0` → pure Ridge
- Good when features are correlated (Lasso alone might randomly pick one from a correlated group)

In [ ]:
# ===========================================================
# Step 5.1: Baseline — Plain Linear Regression (our Phase 5 approach)
# ===========================================================

# TODO: Train a plain Linear Regression model
# - lr = LinearRegression()
# - scores = rmse_cv(lr, X_train_scaled, y_train)
# - results['LinearRegression'] = scores
# - Print: f"Linear Regression RMSE: {scores.mean():.4f} (+/- {scores.std():.4f})"
# - This is our baseline to beat
# - 

In [ ]:
# ===========================================================
# Step 5.2: Ridge Regression
# ===========================================================

# TODO: Train Ridge with different alpha values to find the best one
# - alphas = [0.1, 1.0, 5.0, 10.0, 50.0, 100.0]
# - For each alpha:
#     ridge = Ridge(alpha=alpha)
#     scores = rmse_cv(ridge, X_train_scaled, y_train)
#     print(f"  alpha={alpha:>6}: RMSE = {scores.mean():.4f} (+/- {scores.std():.4f})")

# TODO: Train Ridge with the best alpha
# - best_ridge = Ridge(alpha=<best_alpha>)
# - scores = rmse_cv(best_ridge, X_train_scaled, y_train)
# - results['Ridge'] = scores
# - Print the best score

In [ ]:
# ===========================================================
# Step 5.3: Lasso Regression
# ===========================================================

# TODO: Train Lasso with different alpha values
# - alphas = [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05]
# - Note: Lasso uses much smaller alpha values than Ridge
# - For each alpha:
#     lasso = Lasso(alpha=alpha, max_iter=10000)
#     scores = rmse_cv(lasso, X_train_scaled, y_train)
#     print(f"  alpha={alpha}: RMSE = {scores.mean():.4f} (+/- {scores.std():.4f})")

# TODO: Train Lasso with the best alpha
# - best_lasso = Lasso(alpha=<best_alpha>, max_iter=10000)
# - scores = rmse_cv(best_lasso, X_train_scaled, y_train)
# - results['Lasso'] = scores
# - Print the best score

# TODO: Check how many features Lasso zeroed out
# - best_lasso.fit(X_train_scaled, y_train)
# - n_zero = (best_lasso.coef_ == 0).sum()
# - print(f"Lasso zeroed out {n_zero} of {len(best_lasso.coef_)} features")
# - This is automatic feature selection

In [ ]:
# ===========================================================
# Step 5.4: ElasticNet
# ===========================================================

# TODO: Train ElasticNet with different alpha and l1_ratio combinations
# - alphas = [0.0001, 0.0005, 0.001, 0.005]
# - l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]
# - For each combination:
#     enet = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=10000)
#     scores = rmse_cv(enet, X_train_scaled, y_train)
#     Store and print results

# TODO: Train ElasticNet with the best alpha and l1_ratio
# - best_enet = ElasticNet(alpha=<best_alpha>, l1_ratio=<best_l1_ratio>, max_iter=10000)
# - scores = rmse_cv(best_enet, X_train_scaled, y_train)
# - results['ElasticNet'] = scores
# - Print the best score

---
## Task 6: Random Forest

### Why tree-based models?

Linear models draw a straight line (or hyperplane) through the data. But house prices don't follow straight lines:
- A house with OverallQual=10 isn't just "twice as expensive" as OverallQual=5 — it could be 4x more
- A house with 2 bedrooms and 3 bathrooms is priced differently than the sum of "bedroom effect" + "bathroom effect"

**Decision Trees** solve this by splitting data into regions:
```
IF OverallQual >= 7 AND GrLivArea >= 2000:
    predict $350,000 (average of similar houses)
ELIF OverallQual >= 7 AND GrLivArea < 2000:
    predict $250,000
ELSE:
    predict $150,000
```

**Random Forest** trains many trees (e.g., 500), each on a random subset of data and features, then averages their predictions. This reduces overfitting dramatically.

### Key parameters:

- **`n_estimators`**: Number of trees. More trees = more stable predictions (but slower). 500-1000 is typical.
- **`max_depth`**: How deep each tree can grow. Deeper = more complex patterns, but risk overfitting. `None` lets trees grow fully.
- **`min_samples_leaf`**: Minimum samples in a leaf node. Higher = more conservative trees.
- **`max_features`**: How many features each tree considers at each split. `'sqrt'` means √(total features) — forces diversity.

**Note:** Random Forest does NOT need feature scaling. Trees split on thresholds, so the magnitude of values doesn't matter.

In [ ]:
# TODO: Train a Random Forest model
# - rf = RandomForestRegressor(
#       n_estimators=500,      # 500 trees
#       max_depth=None,        # let trees grow fully
#       min_samples_leaf=2,    # at least 2 samples per leaf
#       max_features='sqrt',   # use sqrt(n_features) at each split
#       random_state=42,
#       n_jobs=-1              # use all CPU cores for speed
#   )

# TODO: Evaluate with cross-validation
# - Note: use X_train (unscaled!) — trees don't need scaling
# - scores = rmse_cv(rf, X_train, y_train)
# - results['RandomForest'] = scores
# - Print: f"Random Forest RMSE: {scores.mean():.4f} (+/- {scores.std():.4f})"

---
## Task 7: XGBoost (Extreme Gradient Boosting)

### How is XGBoost different from Random Forest?

**Random Forest**: Trains trees **independently** in parallel, then averages them.

**XGBoost**: Trains trees **sequentially** — each new tree focuses on correcting the mistakes of all previous trees.

Think of it like this:
1. Tree 1 predicts house prices → gets some wrong
2. Tree 2 is trained on the **errors** of Tree 1 → fixes some mistakes
3. Tree 3 is trained on the **remaining errors** → fixes more
4. After 3000 rounds, the combined prediction is very accurate

This is the same boosting idea as AdaBoost (from Phase 6 Ensemble Learning), but XGBoost uses smarter math (second-order gradient optimization) and is much faster.

### Key parameters:

- **`n_estimators`**: Number of boosting rounds (trees). More rounds = better, but use `early_stopping` to prevent overfitting.
- **`learning_rate`**: How much each tree contributes. Lower (0.01-0.05) = needs more trees but generalizes better.
- **`max_depth`**: Tree depth. Boosting uses shallower trees (3-6) than Random Forest.
- **`subsample`**: Fraction of data used per tree (0.7 = 70%). Randomness prevents overfitting.
- **`colsample_bytree`**: Fraction of features used per tree. Same idea as Random Forest's `max_features`.
- **`reg_alpha`** (L1) and **`reg_lambda`** (L2): Regularization on leaf weights — same idea as Lasso/Ridge but applied inside each tree.

In [ ]:
# TODO: Train an XGBoost model
# - xgb = XGBRegressor(
#       n_estimators=3000,          # up to 3000 boosting rounds
#       learning_rate=0.05,         # small learning rate for careful learning
#       max_depth=4,                # moderately shallow trees
#       subsample=0.7,              # use 70% of data per tree
#       colsample_bytree=0.7,       # use 70% of features per tree
#       reg_alpha=0.1,              # L1 regularization
#       reg_lambda=1.0,             # L2 regularization
#       random_state=42,
#       n_jobs=-1,
#       verbosity=0                 # suppress training output
#   )

# TODO: Evaluate with cross-validation
# - Note: use X_train (unscaled!) — trees don't need scaling
# - scores = rmse_cv(xgb, X_train, y_train)
# - results['XGBoost'] = scores
# - Print: f"XGBoost RMSE: {scores.mean():.4f} (+/- {scores.std():.4f})"

---
## Task 8: LightGBM (Light Gradient Boosting Machine)

### How is LightGBM different from XGBoost?

Both are gradient boosting methods, but LightGBM uses two key innovations:

1. **Leaf-wise growth** (vs XGBoost's level-wise): Instead of growing all leaves at the same depth, LightGBM grows the leaf that reduces error the most. This reaches better accuracy with fewer leaves.

```
XGBoost (level-wise):         LightGBM (leaf-wise):
       O                            O
      / \                          / \
     O   O     (all same depth)   O   O
    / \  / \                     / \
   O  O O  O                    O   O   (grows deepest where it helps most)
```

2. **Histogram-based splitting**: Instead of checking every possible split point, LightGBM groups feature values into bins (histograms). This makes it 10-20x faster than XGBoost on large datasets.

### Key parameters:

- **`n_estimators`**: Number of boosting rounds.
- **`learning_rate`**: Step size shrinkage. Same role as in XGBoost.
- **`num_leaves`**: Maximum leaves per tree. Controls complexity (default: 31). More leaves = more complex.
- **`max_depth`**: Maximum tree depth. Set to -1 for no limit (controlled by `num_leaves` instead).
- **`subsample`** and **`colsample_bytree`**: Same randomness controls as XGBoost.
- **`reg_alpha`** and **`reg_lambda`**: L1/L2 regularization.

In [ ]:
# TODO: Train a LightGBM model
# - lgbm = LGBMRegressor(
#       n_estimators=3000,          # up to 3000 boosting rounds
#       learning_rate=0.05,         # small learning rate
#       num_leaves=31,              # default, controls tree complexity
#       max_depth=4,                # limit depth
#       subsample=0.7,              # use 70% of data per tree
#       colsample_bytree=0.7,       # use 70% of features per tree
#       reg_alpha=0.1,              # L1 regularization
#       reg_lambda=1.0,             # L2 regularization
#       random_state=42,
#       n_jobs=-1,
#       verbosity=-1                # suppress training output
#   )

# TODO: Evaluate with cross-validation
# - Note: use X_train (unscaled!) — trees don't need scaling
# - scores = rmse_cv(lgbm, X_train, y_train)
# - results['LightGBM'] = scores
# - Print: f"LightGBM RMSE: {scores.mean():.4f} (+/- {scores.std():.4f})"

---
## Task 9: Compare All Models

Now let's see which models performed best and why.

In [ ]:
# TODO: Create a comparison table
# - comparison = pd.DataFrame({
#       'Model': results.keys(),
#       'Mean RMSE': [scores.mean() for scores in results.values()],
#       'Std RMSE': [scores.std() for scores in results.values()]
#   })
# - comparison = comparison.sort_values('Mean RMSE')
# - Print the comparison table

# TODO: Visualize model comparison
# - Create a bar plot with model names on x-axis and Mean RMSE on y-axis
# - Add error bars using Std RMSE
# - Color the best model differently
# - Add title: 'Model Comparison (5-Fold CV RMSE on Log-Transformed Target)'
# - Lower RMSE = better

# TODO: Print the winner
# - Print the model with the lowest Mean RMSE

In [ ]:
# TODO: Train all models on full training set and evaluate on holdout test set
#
# For each model, we want to see the actual RMSE on the test set:

# - For linear models (use scaled data): X_train_scaled, X_test_scaled
# - For tree models (use unscaled data): X_train, X_test

# TODO: For each model:
# - model.fit(X_train_appropriate, y_train)
# - y_pred = model.predict(X_test_appropriate)
# - rmse = np.sqrt(mean_squared_error(y_test, y_pred))
# - r2 = r2_score(y_test, y_pred)
# - Print model name, RMSE, and R²

# TODO: Create Actual vs Predicted scatter plot for the best model
# - Convert predictions back to dollar prices: np.expm1(y_pred)
# - Convert actuals back: np.expm1(y_test)
# - Scatter plot with diagonal line
# - Title with model name and R² score

---
## Task 10: Blend the Best Models

### Why blending works

Different models have different strengths:
- **Lasso/Ridge**: Good at capturing linear trends, stable on unseen data
- **XGBoost/LightGBM**: Good at capturing non-linear patterns and interactions

When we average their predictions, the errors of one model get partially cancelled by the other. This almost always improves the score.

**Simple weighted average** is the easiest and most effective approach:

$$\hat{y}_{\text{blend}} = w_1 \cdot \hat{y}_{\text{Ridge}} + w_2 \cdot \hat{y}_{\text{Lasso}} + w_3 \cdot \hat{y}_{\text{XGBoost}} + w_4 \cdot \hat{y}_{\text{LightGBM}}$$

where $w_1 + w_2 + w_3 + w_4 = 1$

**Tip:** Give more weight to models with lower CV RMSE.

In [ ]:
# TODO: Train all models on the full training set
# - best_ridge.fit(X_train_scaled, y_train)
# - best_lasso.fit(X_train_scaled, y_train)
# - xgb.fit(X_train, y_train)
# - lgbm.fit(X_train, y_train)

# TODO: Get predictions from each model on the test set
# - pred_ridge = best_ridge.predict(X_test_scaled)
# - pred_lasso = best_lasso.predict(X_test_scaled)
# - pred_xgb = xgb.predict(X_test)
# - pred_lgbm = lgbm.predict(X_test)

# TODO: Blend predictions with weights
# - Start with equal weights: 0.25 each
# - pred_blend = 0.25 * pred_ridge + 0.25 * pred_lasso + 0.25 * pred_xgb + 0.25 * pred_lgbm

# TODO: Evaluate the blended prediction
# - rmse_blend = np.sqrt(mean_squared_error(y_test, pred_blend))
# - r2_blend = r2_score(y_test, pred_blend)
# - Print: f"Blended RMSE: {rmse_blend:.4f}, R²: {r2_blend:.4f}"
# - Compare with individual model scores

In [ ]:
# TODO: Experiment with different blending weights
# - Try giving more weight to the best-performing models
# - Example: 0.15 * ridge + 0.15 * lasso + 0.35 * xgb + 0.35 * lgbm

# TODO: Try a few weight combinations and track the blended RMSE
# - weight_configs = [
#     (0.25, 0.25, 0.25, 0.25),  # equal
#     (0.15, 0.15, 0.35, 0.35),  # favor tree models
#     (0.10, 0.10, 0.40, 0.40),  # heavily favor trees
#     (0.20, 0.20, 0.30, 0.30),  # slight tree preference
#   ]
# - For each (w1, w2, w3, w4):
#     pred = w1*pred_ridge + w2*pred_lasso + w3*pred_xgb + w4*pred_lgbm
#     rmse = np.sqrt(mean_squared_error(y_test, pred))
#     print(f"Weights ({w1},{w2},{w3},{w4}): RMSE = {rmse:.4f}")

# TODO: Pick the best weight combination and store it
# - best_weights = (<w1>, <w2>, <w3>, <w4>)

---
## Task 11: Generate Kaggle Submission

Now let's produce a real submission file in the format Kaggle expects:
```
Id,SalePrice
1461,169000.1
1462,187724.1
...
```

**Critical steps:**
1. Apply the exact same preprocessing to the test set
2. Make predictions in log-space
3. Inverse-transform with `np.expm1()` to get dollar prices
4. Clip negative predictions to 0 (prices can't be negative)

In [ ]:
# TODO: Load the test data
# - test_df = pd.read_csv('Solutions/test_feature_engineered.csv')
# - test_ids = test_df['Id']  (save IDs for submission)
# - X_kaggle = test_df.drop('Id', axis=1, errors='ignore')
# - Print test_df.shape

# TODO: Apply the same preprocessing as training
# - One-hot encode with same columns:
#   X_kaggle = pd.get_dummies(X_kaggle, columns=cat_cols_from_training, drop_first=True)
# - Align columns with training set (add missing columns as 0, remove extra):
#   X_kaggle = X_kaggle.reindex(columns=X_train.columns, fill_value=0)
# - Fill missing values: X_kaggle = X_kaggle.fillna(0)
# - Scale for linear models: X_kaggle_scaled = pd.DataFrame(scaler.transform(X_kaggle), columns=X_kaggle.columns)

# TODO: Print shapes to verify alignment
# - print(f"Training features: {X_train.shape[1]}")
# - print(f"Test features:     {X_kaggle.shape[1]}")
# - These must match!

In [ ]:
# TODO: Generate blended predictions on Kaggle test set
# - Use the best weights from Task 10
# - (w1, w2, w3, w4) = best_weights
# - pred_kaggle = (
#       w1 * best_ridge.predict(X_kaggle_scaled) +
#       w2 * best_lasso.predict(X_kaggle_scaled) +
#       w3 * xgb.predict(X_kaggle) +
#       w4 * lgbm.predict(X_kaggle)
#   )

# TODO: Inverse log-transform to get actual prices
# - final_predictions = np.expm1(pred_kaggle)
# - final_predictions = np.maximum(final_predictions, 0)  (clip negatives to 0)

# TODO: Create submission DataFrame
# - submission = pd.DataFrame({'Id': test_ids, 'SalePrice': final_predictions})
# - Print submission.head()
# - Print submission.describe() to sanity-check price range

# TODO: Save submission file
# - submission.to_csv('submission_advanced.csv', index=False)
# - Print confirmation and file path

---
## Task 12: Summary and Reflection

### What We Learned

1. **Log-transforming SalePrice** converts RMSLE into RMSE and normalizes the distribution
2. **Regularization** (Ridge, Lasso, ElasticNet) prevents coefficient explosion with many features
3. **Lasso** performs automatic feature selection by zeroing out unimportant features
4. **Tree-based models** (Random Forest, XGBoost, LightGBM) capture non-linear patterns that linear models miss
5. **XGBoost and LightGBM** use sequential boosting — each tree corrects the previous one's mistakes
6. **Cross-validation** gives a reliable performance estimate, not dependent on a single random split
7. **Model blending** combines strengths of different models and nearly always improves the score

### Performance Questions — Answer These:

1. **Improvement**: What was your Phase 5 Kaggle score? What is your new score? How much did it improve?

2. **Best single model**: Which single model performed best in cross-validation? Why do you think that is?

3. **Linear vs Tree**: How much better were XGBoost/LightGBM than Ridge/Lasso? What does this tell you about the data?

4. **Blending benefit**: Did blending improve over the best single model? By how much?

5. **Lasso feature selection**: How many features did Lasso zero out? What does this suggest about the dataset?

6. **Key takeaway**: In what situations would you choose Linear Regression over XGBoost? (Hint: think about interpretability, small datasets, and deployment)

### Write Your Analysis HERE:

(Replace this with your written analysis)

---
## BONUS Task 1: Feature Importance from Tree Models

Tree-based models can tell us which features matter most.

In [ ]:
# TODO: Extract feature importance from XGBoost
# - xgb.fit(X_train, y_train)  (if not already fitted)
# - importance = pd.DataFrame({
#       'Feature': X_train.columns,
#       'Importance': xgb.feature_importances_
#   }).sort_values('Importance', ascending=False)

# TODO: Plot top 20 most important features
# - plt.figure(figsize=(10, 8))
# - top20 = importance.head(20)
# - plt.barh(top20['Feature'], top20['Importance'], color='teal')
# - plt.xlabel('Feature Importance')
# - plt.title('Top 20 Features (XGBoost)')
# - plt.gca().invert_yaxis()  (highest importance at top)
# - plt.tight_layout()
# - plt.show()

# TODO: Compare with Lasso's feature selection
# - Which features appear as important in both XGBoost and Lasso?
# - Print the overlap

---
## BONUS Task 2: Hyperparameter Tuning with GridSearchCV

Fine-tune the best model for even better performance.

In [ ]:
# TODO: Import GridSearchCV
# - from sklearn.model_selection import GridSearchCV

# TODO: Define parameter grid for LightGBM (or XGBoost)
# - param_grid = {
#       'n_estimators': [1000, 2000, 3000],
#       'learning_rate': [0.01, 0.05, 0.1],
#       'max_depth': [3, 4, 5],
#       'num_leaves': [15, 31, 63],
#   }
#
# Note: This grid has 3*3*3*3 = 81 combinations × 5 folds = 405 model trainings
# If too slow, reduce the grid or use RandomizedSearchCV instead

# TODO: Run GridSearchCV
# - grid_search = GridSearchCV(
#       LGBMRegressor(random_state=42, n_jobs=-1, verbosity=-1),
#       param_grid,
#       cv=kf,
#       scoring='neg_root_mean_squared_error',
#       verbose=1
#   )
# - grid_search.fit(X_train, y_train)

# TODO: Print best parameters and score
# - print(f"Best Parameters: {grid_search.best_params_}")
# - print(f"Best CV RMSE: {-grid_search.best_score_:.4f}")

# TODO: Use the tuned model in your blend and re-evaluate

---
## BONUS Task 3: Stacking (Advanced Ensemble)

Stacking goes beyond simple averaging: it trains a **meta-model** that learns the optimal way to combine base model predictions.

```
Base Models:          Meta-Model:
Ridge    → pred_1 ─┐
Lasso    → pred_2 ─┤→ LinearRegression → final prediction
XGBoost  → pred_3 ─┤
LightGBM → pred_4 ─┘
```

In [ ]:
# TODO: Import StackingRegressor
# - from sklearn.ensemble import StackingRegressor

# TODO: Create the stacking ensemble
# - stacker = StackingRegressor(
#       estimators=[
#           ('ridge', best_ridge),
#           ('lasso', best_lasso),
#           ('xgb', xgb),
#           ('lgbm', lgbm)
#       ],
#       final_estimator=Ridge(alpha=10.0),  # meta-model
#       cv=kf
#   )
#
# Note: stacker uses scaled data for linear base models and unscaled for trees.
# Since StackingRegressor passes the same data to all estimators,
# you may want to use only tree models here, or use a Pipeline for scaling.
# Alternatively, use the simple blending approach from Task 10.

# TODO: Evaluate stacker
# - scores = rmse_cv(stacker, X_train, y_train)
# - print(f"Stacking RMSE: {scores.mean():.4f} (+/- {scores.std():.4f})")
# - Compare with simple blending from Task 10

---
## Submission Checklist

Make sure your notebook includes:

- ✓ All required libraries imported (including xgboost, lightgbm)
- ✓ Data loaded with log-transformed target
- ✓ Features preprocessed (encoded, scaled, no missing values)
- ✓ Cross-validation framework set up and used consistently
- ✓ Ridge, Lasso, ElasticNet trained with alpha tuning
- ✓ Random Forest trained
- ✓ XGBoost trained with proper hyperparameters
- ✓ LightGBM trained with proper hyperparameters
- ✓ All models compared in a table and chart
- ✓ Models blended with optimized weights
- ✓ Kaggle submission file generated (submission_advanced.csv)
- ✓ All reflection questions answered
- ✓ Clear documentation and comments throughout

## Key Takeaways

1. **Log-transforming the target** is essential when Kaggle uses RMSLE scoring
2. **Regularization** (Ridge/Lasso) prevents overfitting when you have many features
3. **Lasso** doubles as a feature selector — it tells you which of your 200+ features actually matter
4. **Tree-based models** (XGBoost, LightGBM) capture non-linear patterns that linear models fundamentally cannot
5. **Cross-validation** replaces unreliable single-split evaluation
6. **Model blending** is the single most effective technique in competitive ML
7. **No single model is always best** — the strength of an ensemble is that different models compensate for each other's weaknesses